# AED v7 — split global, macro-F1 e hard negatives

Esta versão mantém `YAMNet + MFCC`, mas corrige os problemas do AED v6:

- split global por grupo, sem dividir o mesmo ator entre classes;
- grupos AOB incluem a pasta de origem e não colidem;
- variantes TAU da mesma gravação ficam juntas;
- treinamento é bloqueado se existir qualquer vazamento;
- amostragem balanceada, com prioridade adicional para `impacto`;
- checkpoint monitorado por `val_evento_macro_f1`;
- arquitetura mais regularizada para reduzir overfitting;
- segunda etapa com mineração de ambientes normais confundidos com perigo;
- seleção automática entre etapa 1 e etapa 2;
- avaliação externa considerando apenas classes realmente presentes;
- verificação de paridade Keras × TFLite.

Meta experimental:

- acurácia de teste `>= 90%`;
- F1 macro de teste `>= 90%`;
- menor F1 por classe `>= 80%`.

A meta não é garantida. O resultado só é aceito se o teste limpo atingir os critérios.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install             kagglehub             librosa==0.11.0             soundfile             scikit-learn             pandas             matplotlib             tqdm             pyyaml             requests             tensorflow-hub

In [ ]:
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import gc
import hashlib
import json
import math
import random
import re
import shutil
import zipfile

import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub

from google.colab import files
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "Ative uma GPU T4 no Colab."
    )

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass

tf.keras.mixed_precision.set_global_policy(
    "mixed_float16"
)

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    sr: int = 16000
    duracao: float = 2.0
    passo: float = 0.50

    n_mfcc: int = 40
    n_fft: int = 512
    hop_length: int = 256

    max_arquivos_por_classe: int = 5000
    max_janelas_por_arquivo: int = 4

    lote: int = 64
    epocas_etapa_1: int = 70
    epocas_etapa_2: int = 28
    paciencia: int = 10
    seed: int = 42

    tentativas_split: int = 350
    max_hard_negatives: int = 1600
    repeticoes_hard_negative: int = 4
    limiar_hard_negative: float = 0.22

    meta_accuracy: float = 0.90
    meta_f1_macro: float = 0.90
    meta_f1_classe: float = 0.80
    meta_paridade: float = 0.99

    usar_aob: bool = True
    usar_tau_privacy: bool = True
    usar_tut: bool = True
    usar_reforco_manual: bool = False


CFG = Configuracao()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {
    classe: indice
    for indice, classe
    in enumerate(CLASSES)
}

FAMILIA = {
    "ambiente_normal": 0,
    "grito_panico": 1,
    "choro_distress": 1,
    "impacto": 2,
    "vidro_quebrando": 2,
    "tiro": 2,
}

RARAS = {
    "impacto",
    "vidro_quebrando",
    "tiro",
}

AUGMENTACOES = {
    "ambiente_normal": 0,
    "grito_panico": 0,
    "choro_distress": 1,
    "impacto": 4,
    "vidro_quebrando": 3,
    "tiro": 2,
}

PESOS_AMOSTRAGEM = {
    "ambiente_normal": 0.22,
    "grito_panico": 0.14,
    "choro_distress": 0.14,
    "impacto": 0.18,
    "vidro_quebrando": 0.16,
    "tiro": 0.16,
}

if not np.isclose(sum(PESOS_AMOSTRAGEM.values()), 1.0):
    raise RuntimeError("Pesos de amostragem inválidos.")

RAIZ = Path("/content/aed_v7")
DADOS = RAIZ / "datasets"
CACHE = RAIZ / "cache"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

ESC_DIR = DADOS / "esc50"
TUT_DIR = DADOS / "tut_rare"
AOB_DEV = DADOS / "aob_dev"
AOB_EVAL = DADOS / "aob_eval"
TAU_DIR = DADOS / "tau_privacy"
REFORCO = DADOS / "reforco_manual"

for pasta in (
    RAIZ,
    DADOS,
    CACHE,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Download dos datasets

In [ ]:
ASVP = Path(
    kagglehub.dataset_download(
        (
            "dejolilandry/"
            "asvpesdspeech-nonspeech-emotional-utterances"
        )
    )
)

GUNSHOT = Path(
    kagglehub.dataset_download(
        "emrahaydemr/gunshot-audio-dataset"
    )
)

URBAN = Path(
    kagglehub.dataset_download(
        "chrisfilo/urbansound8k"
    )
)

SCREAM = Path(
    kagglehub.dataset_download(
        (
            "aananehsansiam/"
            "audio-dataset-of-scream-and-non-scream"
        )
    )
)

print("ASVP:", ASVP)
print("Gunshot:", GUNSHOT)
print("UrbanSound8K:", URBAN)
print("Scream:", SCREAM)

In [ ]:
import time


def zip_valido(
    caminho: Path,
) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip(
    urls: str | list[str],
    destino: Path,
    obrigatorio: bool = True,
    tentativas_por_url: int = 4,
) -> bool:
    if zip_valido(destino):
        print("Já disponível:", destino)
        return True

    if isinstance(urls, str):
        urls = [urls]

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )

    # Pode ter terminado o download, mas não ter sido renomeado.
    if zip_valido(temporario):
        temporario.replace(destino)
        print("Download parcial já estava completo:", destino)
        return True

    ultimo_erro = None

    for url in urls:
        for tentativa in range(
            1,
            tentativas_por_url + 1,
        ):
            try:
                tamanho_existente = (
                    temporario.stat().st_size
                    if temporario.exists()
                    else 0
                )

                cabecalhos = {
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "violence-edge-ai-aed-v7"
                    )
                }

                if tamanho_existente > 0:
                    cabecalhos["Range"] = (
                        f"bytes={tamanho_existente}-"
                    )

                print(
                    "\nBaixando:",
                    destino.name,
                )
                print(
                    "Rota:",
                    url,
                )
                print(
                    f"Tentativa: {tentativa}/"
                    f"{tentativas_por_url}",
                )
                print(
                    "Já baixado:",
                    f"{tamanho_existente / 1024**2:.1f} MB",
                )

                with requests.get(
                    url,
                    stream=True,
                    timeout=(45, 600),
                    verify=True,
                    allow_redirects=True,
                    headers=cabecalhos,
                ) as resposta:
                    # O servidor pode informar que o arquivo já terminou.
                    if (
                        resposta.status_code == 416
                        and zip_valido(temporario)
                    ):
                        temporario.replace(destino)
                        return True

                    resposta.raise_for_status()

                    # 206 significa que o servidor aceitou continuar.
                    continuar = (
                        tamanho_existente > 0
                        and resposta.status_code == 206
                    )

                    if continuar:
                        modo = "ab"
                        inicio_barra = tamanho_existente
                    else:
                        modo = "wb"
                        inicio_barra = 0

                    restante = int(
                        resposta.headers.get(
                            "content-length",
                            0,
                        )
                    )

                    total = (
                        inicio_barra + restante
                        if restante > 0
                        else None
                    )

                    with temporario.open(
                        modo
                    ) as arquivo:
                        with tqdm(
                            total=total,
                            initial=inicio_barra,
                            unit="B",
                            unit_scale=True,
                            desc=destino.name,
                        ) as barra:
                            for bloco in resposta.iter_content(
                                chunk_size=4 * 1024 * 1024
                            ):
                                if not bloco:
                                    continue

                                arquivo.write(bloco)
                                barra.update(len(bloco))

                if not zip_valido(temporario):
                    raise RuntimeError(
                        "O arquivo recebido não é um ZIP válido."
                    )

                temporario.replace(destino)

                print(
                    "Download concluído:",
                    destino,
                )

                return True

            except Exception as erro:
                ultimo_erro = erro

                print(
                    "Falhou:",
                    type(erro).__name__,
                    erro,
                )

                # Mantém o .part para tentar continuar depois.
                espera = min(
                    60,
                    5 * (2 ** (tentativa - 1)),
                )

                print(
                    f"Nova tentativa em {espera}s..."
                )

                time.sleep(espera)

    mensagem = (
        f"Não foi possível baixar {destino.name}. "
        f"Último erro: {ultimo_erro}"
    )

    if obrigatorio:
        raise RuntimeError(mensagem)

    print("\nAVISO:", mensagem)
    print(
        "Este dataset será ignorado e "
        "o notebook continuará."
    )

    return False


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> bool:
    if not zip_valido(arquivo_zip):
        print(
            "Extração ignorada; ZIP ausente:",
            arquivo_zip,
        )
        return False

    marcador = destino / ".extraido"

    if marcador.exists():
        print(
            "Já extraído:",
            destino,
        )
        return True

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Extraindo:",
        arquivo_zip.name,
    )

    with zipfile.ZipFile(
        arquivo_zip
    ) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()

    print(
        "Extração concluída:",
        destino,
    )

    return True


# ============================================================
# ESC-50 — obrigatório
# ============================================================

ZIP_ESC = DADOS / "esc50.zip"

esc_disponivel = baixar_zip(
    [
        (
            "https://github.com/karolpiczak/"
            "ESC-50/archive/refs/heads/master.zip"
        ),
    ],
    ZIP_ESC,
    obrigatorio=True,
)

if esc_disponivel:
    extrair_zip(
        ZIP_ESC,
        ESC_DIR,
    )


# ============================================================
# TUT Rare — opcional
# O notebook continua caso o Zenodo permaneça indisponível.
# ============================================================

TUT_DISPONIVEL = False

if CFG.usar_tut:
    ZIP_TUT = DADOS / "tut_rare.zip"

    nome_tut = (
        "TUT-rare-sound-events-2017-development."
        "source_data_events.zip"
    )

    TUT_DISPONIVEL = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/401395/"
                f"files/{nome_tut}/content"
            ),
            (
                "https://zenodo.org/records/401395/files/"
                f"{nome_tut}?download=1"
            ),
        ],
        ZIP_TUT,
        obrigatorio=False,
        tentativas_por_url=4,
    )

    if TUT_DISPONIVEL:
        extrair_zip(
            ZIP_TUT,
            TUT_DIR,
        )

print(
    "TUT disponível:",
    TUT_DISPONIVEL,
)


# ============================================================
# Dataset-AOB — obrigatório
# ============================================================

if CFG.usar_aob:
    ZIP_AOB_DEV = (
        DADOS / "aob_dev.zip"
    )

    ZIP_AOB_EVAL = (
        DADOS / "aob_eval.zip"
    )

    aob_dev_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/4319802/"
                "files/DATASET_AOB.zip/content"
            ),
            (
                "https://zenodo.org/records/4319802/files/"
                "DATASET_AOB.zip?download=1"
            ),
        ],
        ZIP_AOB_DEV,
        obrigatorio=True,
    )

    aob_eval_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/4319802/"
                "files/EVALUATION_dataset_AOB.zip/content"
            ),
            (
                "https://zenodo.org/records/4319802/files/"
                "EVALUATION_dataset_AOB.zip?download=1"
            ),
        ],
        ZIP_AOB_EVAL,
        obrigatorio=True,
    )

    if aob_dev_disponivel:
        extrair_zip(
            ZIP_AOB_DEV,
            AOB_DEV,
        )

    if aob_eval_disponivel:
        extrair_zip(
            ZIP_AOB_EVAL,
            AOB_EVAL,
        )


# ============================================================
# TAU Privacy — obrigatório
# ============================================================

if CFG.usar_tau_privacy:
    ZIP_TAU = (
        DADOS / "tau_privacy.zip"
    )

    tau_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/10378145/"
                "files/final_data_wav_split.zip/content"
            ),
            (
                "https://zenodo.org/records/10378145/files/"
                "final_data_wav_split.zip?download=1"
            ),
        ],
        ZIP_TAU,
        obrigatorio=True,
    )

    if tau_disponivel:
        extrair_zip(
            ZIP_TAU,
            TAU_DIR,
        )


print("\nDownloads concluídos.")
print("ESC-50:", esc_disponivel)
print("TUT Rare:", TUT_DISPONIVEL)

if CFG.usar_aob:
    print("AOB desenvolvimento:", aob_dev_disponivel)
    print("AOB avaliação:", aob_eval_disponivel)

if CFG.usar_tau_privacy:
    print("TAU Privacy:", tau_disponivel)

## 3. Reforço manual opcional

## 4. Metadados unificados

In [ ]:
EXTENSOES = {
    ".wav",
    ".mp3",
    ".ogg",
    ".flac",
    ".m4a",
}


def audios_em(
    pasta: Path | None,
) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES
        )
    )


def token(texto: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def base_gravacao(nome: str) -> str:
    nome = token(nome)
    nome = re.sub(
        r"_(speech|no_speech|nospeech).*$",
        "",
        nome,
    )
    nome = re.sub(
        r"_(chunk|segment|slice|part)_?\d+$",
        "",
        nome,
    )
    return nome or "gravacao"


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    split_fixo: str = "",
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)

    if arquivo.exists() and classe in ID:
        registros.append(
            {
                "arquivo": str(arquivo.resolve()),
                "classe": classe,
                "classe_id": ID[classe],
                "familia": FAMILIA[classe],
                "perigo": int(
                    classe != "ambiente_normal"
                ),
                "fonte": fonte,
                "grupo": str(grupo),
                "split_fixo": split_fixo,
                "extra": str(extra),
            }
        )


registros = []


# ASVP
mapa_asvp = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "06": "grito_panico",
}

for arquivo in audios_em(ASVP):
    numeros = re.findall(
        r"\d+",
        arquivo.stem,
    )

    if len(numeros) < 6:
        continue

    emocao = numeros[2].zfill(2)
    ator = numeros[5].zfill(3)
    classe = mapa_asvp.get(emocao)

    if classe is None:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{ator}",
        extra=emocao,
    )


# Scream / non-scream
for arquivo in audios_em(SCREAM):
    relativo = arquivo.relative_to(
        SCREAM
    )

    pastas = [
        token(parte)
        for parte in relativo.parts[:-1]
    ]

    if any(
        nome in {
            "non_scream",
            "nonscream",
            "no_scream",
            "not_scream",
        }
        for nome in pastas
    ):
        classe = "ambiente_normal"

    elif any(
        nome in {
            "scream",
            "screams",
        }
        for nome in pastas
    ):
        classe = "grito_panico"

    else:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ScreamDataset",
        (
            f"scream:{classe}:"
            f"{token(arquivo.stem)}"
        ),
        extra="/".join(pastas),
    )


# Gunshot
for arquivo in audios_em(GUNSHOT):
    relativo = arquivo.relative_to(
        GUNSHOT
    )

    origem = token(
        relativo.parts[0]
        if relativo.parts
        else arquivo.parent.name
    )

    sessao = re.sub(
        r"\s*\(\d+\)$",
        "",
        arquivo.stem,
    )
    sessao = token(sessao)

    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:{origem}:{sessao}",
        extra=origem,
    )


# ESC-50
csvs_esc = list(
    ESC_DIR.rglob("esc50.csv")
)
pastas_esc = [
    pasta
    for pasta in ESC_DIR.rglob("audio")
    if pasta.is_dir()
]

if not csvs_esc or not pastas_esc:
    raise RuntimeError(
        "ESC-50 incompleto."
    )

tabela_esc = pd.read_csv(
    csvs_esc[0]
)
audio_esc = pastas_esc[0]

mapa_esc = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}

normais_esc = {
    "clapping",
    "hand_saw",
    "door_wood_creaks",
    "fireworks",
    "chainsaw",
    "siren",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "clock_alarm",
    "can_opening",
    "mouse_click",
    "keyboard_typing",
    "footsteps",
    "laughing",
    "breathing",
    "coughing",
    "sneezing",
    "snoring",
    "rain",
    "sea_waves",
    "wind",
    "thunderstorm",
    "airplane",
    "helicopter",
    "train",
}

for linha in tabela_esc.itertuples():
    categoria = str(linha.category)

    if categoria in mapa_esc:
        classe = mapa_esc[categoria]
    elif categoria in normais_esc:
        classe = "ambiente_normal"
    else:
        continue

    adicionar(
        registros,
        audio_esc / str(linha.filename),
        classe,
        "ESC-50",
        f"esc50:{linha.src_file}",
        extra=categoria,
    )


# UrbanSound8K
csvs_urban = list(
    URBAN.rglob("UrbanSound8K.csv")
)

if csvs_urban:
    tabela_urban = pd.read_csv(
        csvs_urban[0]
    )
    raiz_audio = (
        csvs_urban[0].parent.parent
        / "audio"
    )

    for _, linha in tabela_urban.iterrows():
        nome = str(linha["class"])

        if nome == "gun_shot":
            classe = "tiro"
        elif nome in {
            "jackhammer",
            "drilling",
            "siren",
            "car_horn",
            "dog_bark",
            "street_music",
            "children_playing",
            "air_conditioner",
            "engine_idling",
        }:
            classe = "ambiente_normal"
        else:
            continue

        arquivo = (
            raiz_audio
            / f"fold{int(linha['fold'])}"
            / str(linha["slice_file_name"])
        )

        fsid = str(
            linha.get(
                "fsID",
                str(
                    linha["slice_file_name"]
                ).split("-")[0],
            )
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "UrbanSound8K",
            f"urban:fsid_{fsid}",
            extra=nome,
        )


# TUT Rare
if CFG.usar_tut:
    for arquivo in audios_em(TUT_DIR):
        texto = token(str(arquivo))

        if any(
            termo in texto
            for termo in (
                "babycry",
                "baby_cry",
                "babycrying",
            )
        ):
            classe = "choro_distress"
        elif any(
            termo in texto
            for termo in (
                "glassbreak",
                "glass_break",
                "glassbreaking",
            )
        ):
            classe = "vidro_quebrando"
        elif any(
            termo in texto
            for termo in (
                "gunshot",
                "gun_shot",
                "gunfire",
            )
        ):
            classe = "tiro"
        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "TUT_Rare",
            f"tut:{classe}:{arquivo.stem}",
            extra=classe,
        )


# Dataset-AOB
def classe_aob(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if any(
        termo in texto
        for termo in (
            "glass_breaking",
            "glassbreaking",
            "glass_break",
        )
    ):
        return "vidro_quebrando"

    if any(
        termo in texto
        for termo in (
            "gun_shot",
            "gunshot",
        )
    ):
        return "tiro"

    if any(
        termo in texto
        for termo in (
            "screams",
            "scream",
        )
    ):
        return "grito_panico"

    if any(
        termo in texto
        for termo in (
            "alarm_siren",
            "children_playing",
            "dog_bark",
            "engine",
            "footsteps",
            "metro_train",
            "rain",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_aob:
    for raiz_aob, split_fixo in (
        (AOB_DEV, ""),
        (AOB_EVAL, "externo"),
    ):
        for arquivo in audios_em(
            raiz_aob
        ):
            classe = classe_aob(
                arquivo
            )

            if classe is None:
                continue

            relativo = arquivo.relative_to(
                raiz_aob
            )

            categoria = token(
                str(relativo.parent)
            )
            grupo_base = base_gravacao(
                relativo.stem
            )

            adicionar(
                registros,
                arquivo,
                classe,
                "Dataset-AOB",
                (
                    f"aob:{split_fixo or 'dev'}:"
                    f"{categoria}:{grupo_base}"
                ),
                split_fixo=split_fixo,
                extra=str(relativo.parent),
            )


# TAU Privacy
def classe_tau(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if "glass_breaking" in texto:
        return "vidro_quebrando"

    if (
        "gun_shot" in texto
        or "gunshot" in texto
    ):
        return "tiro"

    if "slam" in texto:
        return "impacto"

    if "crying" in texto:
        return "choro_distress"

    if any(
        termo in texto
        for termo in (
            "dog_barking",
            "cough",
            "applause",
            "dished_pot_pan",
            "toilet_flush",
            "cat_meowing",
            "doorbell",
            "drill",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_tau_privacy:
    for arquivo in audios_em(TAU_DIR):
        classe = classe_tau(arquivo)

        if classe is None:
            continue

        relativo = arquivo.relative_to(
            TAU_DIR
        )
        evento = token(
            relativo.parent.parent.name
            if len(relativo.parts) >= 3
            else classe
        )
        base = base_gravacao(
            arquivo.stem
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "TAU-Privacy",
            f"tau:{evento}:{base}",
            extra=str(
                arquivo.relative_to(
                    TAU_DIR
                ).parent
            ),
        )


# Reforço manual
if CFG.usar_reforco_manual:
    for classe in CLASSES:
        for pasta in REFORCO.rglob(
            classe
        ):
            if not pasta.is_dir():
                continue

            for arquivo in audios_em(
                pasta
            ):
                adicionar(
                    registros,
                    arquivo,
                    classe,
                    "ReforcoManual",
                    (
                        f"manual:{classe}:"
                        f"{token(arquivo.stem)}"
                    ),
                    extra=classe,
                )


meta = pd.DataFrame(registros)

if meta.empty:
    raise RuntimeError(
        "Nenhum áudio encontrado."
    )

meta = meta.drop_duplicates(
    subset=["arquivo", "classe"]
)

desenvolvimento = meta[
    meta["split_fixo"] != "externo"
].copy()

externo = meta[
    meta["split_fixo"] == "externo"
].copy()

partes = []

for classe in CLASSES:
    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ].copy()

    if (
        len(parte)
        > CFG.max_arquivos_por_classe
    ):
        parte = parte.sample(
            CFG.max_arquivos_por_classe,
            random_state=(
                CFG.seed + ID[classe]
            ),
        )

    partes.append(parte)

desenvolvimento = pd.concat(
    partes,
    ignore_index=True,
)

display(
    desenvolvimento.groupby(
        ["classe", "fonte"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

resumo = (
    desenvolvimento.groupby(
        "classe"
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo)

faltantes = [
    classe
    for classe in CLASSES
    if resumo.loc[
        classe,
        "arquivos",
    ] == 0
]

if faltantes:
    raise RuntimeError(
        f"Classes sem dados: {faltantes}"
    )

desenvolvimento.to_csv(
    RESULTADOS
    / "fontes_desenvolvimento.csv",
    index=False,
)

externo.to_csv(
    RESULTADOS
    / "fontes_externo_aob.csv",
    index=False,
)

## 5. Auditoria manual

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ]

    for linha in (
        parte.sample(
            min(2, len(parte)),
            random_state=(
                CFG.seed + ID[classe]
            ),
        )
        .itertuples()
    ):
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            linha.extra,
        )
        display(
            Audio(linha.arquivo)
        )

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras e altere "
        "APROVAR_AUDITORIA para True."
    )

## 6. Split global por grupo, sem vazamento

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
def pontuar_split(tabela: pd.DataFrame) -> float:
    alvos = {
        "treino": 0.70,
        "validacao": 0.15,
        "teste": 0.15,
    }

    total_classe = (
        tabela.groupby("classe")
        .size()
        .reindex(CLASSES, fill_value=0)
    )

    score = 0.0

    for split, alvo in alvos.items():
        parte = tabela[
            tabela["split"] == split
        ]

        score += abs(
            len(parte) / len(tabela) - alvo
        )

        contagem = (
            parte.groupby("classe")
            .size()
            .reindex(CLASSES, fill_value=0)
        )

        proporcao = (
            contagem
            / total_classe.replace(0, np.nan)
        ).fillna(0.0)

        score += float(
            np.abs(proporcao - alvo).mean()
        )

    return score


def criar_split_global(
    tabela: pd.DataFrame,
):
    melhor = None
    melhor_score = float("inf")

    for tentativa in range(
        CFG.tentativas_split
    ):
        semente = CFG.seed + tentativa

        divisor_1 = GroupShuffleSplit(
            n_splits=1,
            train_size=0.70,
            random_state=semente,
        )

        idx_treino, idx_tmp = next(
            divisor_1.split(
                tabela,
                groups=tabela["grupo"],
            )
        )

        temporario = tabela.iloc[idx_tmp]

        divisor_2 = GroupShuffleSplit(
            n_splits=1,
            train_size=0.50,
            random_state=semente + 10000,
        )

        idx_val_rel, idx_teste_rel = next(
            divisor_2.split(
                temporario,
                groups=temporario["grupo"],
            )
        )

        candidato = tabela.copy()
        candidato["split"] = ""
        candidato.iloc[
            idx_treino,
            candidato.columns.get_loc("split"),
        ] = "treino"

        indices_val = temporario.index[
            idx_val_rel
        ]
        indices_teste = temporario.index[
            idx_teste_rel
        ]

        candidato.loc[
            indices_val,
            "split",
        ] = "validacao"
        candidato.loc[
            indices_teste,
            "split",
        ] = "teste"

        distribuicao = (
            candidato.groupby(
                ["classe", "split"]
            )
            .size()
            .unstack(fill_value=0)
            .reindex(CLASSES, fill_value=0)
        )

        esperadas = [
            "treino",
            "validacao",
            "teste",
        ]

        if any(
            split not in distribuicao.columns
            for split in esperadas
        ):
            continue

        if (
            distribuicao[esperadas]
            .eq(0)
            .any()
            .any()
        ):
            continue

        score = pontuar_split(candidato)

        if score < melhor_score:
            melhor = candidato
            melhor_score = score

    if melhor is None:
        raise RuntimeError(
            "Não foi possível criar um split válido."
        )

    grupos = {
        split: set(
            melhor.loc[
                melhor["split"] == split,
                "grupo",
            ]
        )
        for split in (
            "treino",
            "validacao",
            "teste",
        )
    }

    intersecoes = {
        "treino_validacao": len(
            grupos["treino"]
            & grupos["validacao"]
        ),
        "treino_teste": len(
            grupos["treino"]
            & grupos["teste"]
        ),
        "validacao_teste": len(
            grupos["validacao"]
            & grupos["teste"]
        ),
    }

    if any(intersecoes.values()):
        raise RuntimeError(
            f"Vazamento entre splits: {intersecoes}"
        )

    return melhor, {
        "score": float(melhor_score),
        "intersecoes": intersecoes,
        "grupos_por_split": {
            split: len(valor)
            for split, valor in grupos.items()
        },
    }


meta_split, auditoria_split = (
    criar_split_global(
        desenvolvimento
    )
)

display(
    meta_split.groupby(
        ["classe", "split"]
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
)

print(
    json.dumps(
        auditoria_split,
        indent=2,
    )
)

meta_split.to_csv(
    RESULTADOS / "fontes_com_split.csv",
    index=False,
)

(
    RESULTADOS / "auditoria_split.json"
).write_text(
    json.dumps(
        auditoria_split,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 7. Extrair YAMNet + MFCC

In [ ]:
YAMNET_HANDLE = (
    "https://tfhub.dev/google/yamnet/1"
)

yamnet = hub.load(
    YAMNET_HANDLE
)

AMOSTRAS = int(
    CFG.sr * CFG.duracao
)


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.sr,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    pico = float(
        np.max(np.abs(audio))
    )

    if pico > 1.0:
        audio = audio / pico

    return audio


def ajustar(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS - len(audio),
            ),
        )

    return audio[:AMOSTRAS]


def janelas_uniformes(
    audio: np.ndarray,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    inicios = np.linspace(
        0,
        len(audio) - AMOSTRAS,
        CFG.max_janelas_por_arquivo,
        dtype=int,
    )

    return [
        audio[
            inicio:
            inicio + AMOSTRAS
        ]
        for inicio in inicios
    ]


def janelas_onset(
    audio: np.ndarray,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    onset = librosa.onset.onset_strength(
        y=audio,
        sr=CFG.sr,
        hop_length=CFG.hop_length,
    )

    picos = librosa.util.peak_pick(
        onset,
        pre_max=3,
        post_max=3,
        pre_avg=5,
        post_avg=5,
        delta=0.20,
        wait=8,
    )

    if len(picos) == 0:
        return janelas_uniformes(audio)

    ordem = np.argsort(
        onset[picos]
    )[::-1]

    janelas = []
    usados = []

    for indice in ordem:
        centro = librosa.frames_to_samples(
            int(picos[indice]),
            hop_length=CFG.hop_length,
        )

        inicio = int(
            np.clip(
                centro - AMOSTRAS // 2,
                0,
                len(audio) - AMOSTRAS,
            )
        )

        if any(
            abs(inicio - anterior)
            < CFG.sr * 0.30
            for anterior in usados
        ):
            continue

        janelas.append(
            audio[
                inicio:
                inicio + AMOSTRAS
            ]
        )
        usados.append(inicio)

        if (
            len(janelas)
            >= CFG.max_janelas_por_arquivo
        ):
            break

    return (
        janelas
        if janelas
        else janelas_uniformes(audio)
    )


def augmentar_waveform(
    audio: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    audio = audio.copy()

    ganho_db = rng.uniform(
        -6.0,
        5.0,
    )

    audio *= 10 ** (
        ganho_db / 20.0
    )

    deslocamento = int(
        rng.uniform(
            -0.15,
            0.15,
        )
        * CFG.sr
    )

    audio = np.roll(
        audio,
        deslocamento,
    )

    ruido = rng.normal(
        0.0,
        rng.uniform(
            0.001,
            0.012,
        ),
        size=audio.shape,
    ).astype(np.float32)

    audio = audio + ruido

    return np.clip(
        audio,
        -1.0,
        1.0,
    )


def extrair_vetor(
    audio: np.ndarray,
) -> np.ndarray:
    audio = ajustar(audio).astype(
        np.float32
    )

    _, embeddings, _ = yamnet(
        tf.convert_to_tensor(
            audio,
            dtype=tf.float32,
        )
    )

    emb = embeddings.numpy()

    yamnet_stats = np.concatenate(
        [
            emb.mean(axis=0),
            emb.std(axis=0),
            emb.max(axis=0),
        ]
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=CFG.sr,
        n_mfcc=CFG.n_mfcc,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )

    delta = librosa.feature.delta(
        mfcc,
        width=9,
    )

    delta2 = librosa.feature.delta(
        mfcc,
        order=2,
        width=9,
    )

    mfcc_stats = np.concatenate(
        [
            mfcc.mean(axis=1),
            mfcc.std(axis=1),
            delta.mean(axis=1),
            delta.std(axis=1),
            delta2.mean(axis=1),
            delta2.std(axis=1),
        ]
    )

    espectral = np.asarray(
        [
            np.mean(
                librosa.feature.rms(
                    y=audio
                )
            ),
            np.std(
                librosa.feature.rms(
                    y=audio
                )
            ),
            np.mean(
                librosa.feature.spectral_centroid(
                    y=audio,
                    sr=CFG.sr,
                )
            ),
            np.std(
                librosa.feature.spectral_centroid(
                    y=audio,
                    sr=CFG.sr,
                )
            ),
            np.mean(
                librosa.feature.zero_crossing_rate(
                    audio
                )
            ),
            np.std(
                librosa.feature.zero_crossing_rate(
                    audio
                )
            ),
            np.max(np.abs(audio)),
        ],
        dtype=np.float32,
    )

    return np.concatenate(
        [
            yamnet_stats,
            mfcc_stats,
            espectral,
        ]
    ).astype(np.float32)


registros_features = []
vetores = []

rng = np.random.default_rng(
    CFG.seed
)

for linha in tqdm(
    meta_split.itertuples(),
    total=len(meta_split),
    desc="Extraindo features",
):
    try:
        audio = carregar_audio(
            linha.arquivo
        )

        if linha.classe in RARAS:
            janelas = janelas_onset(
                audio
            )
        else:
            janelas = janelas_uniformes(
                audio
            )

    except Exception as erro:
        print(
            "Falha:",
            linha.arquivo,
            erro,
        )
        continue

    for indice, janela in enumerate(
        janelas
    ):
        vetores.append(
            extrair_vetor(janela)
        )

        registros_features.append(
            {
                "classe": linha.classe,
                "classe_id": linha.classe_id,
                "familia": linha.familia,
                "perigo": linha.perigo,
                "grupo": linha.grupo,
                "split": linha.split,
                "fonte": linha.fonte,
                "arquivo": linha.arquivo,
                "augmentado": False,
            }
        )

        quantidade_aug = (
            AUGMENTACOES[linha.classe]
            if linha.split == "treino"
            else 0
        )

        for _ in range(
            quantidade_aug
        ):
            aumentado = augmentar_waveform(
                janela,
                rng,
            )

            vetores.append(
                extrair_vetor(
                    aumentado
                )
            )

            registros_features.append(
                {
                    "classe": linha.classe,
                    "classe_id": linha.classe_id,
                    "familia": linha.familia,
                    "perigo": linha.perigo,
                    "grupo": linha.grupo,
                    "split": linha.split,
                    "fonte": linha.fonte,
                    "arquivo": linha.arquivo,
                    "augmentado": True,
                }
            )

X = np.asarray(
    vetores,
    dtype=np.float32,
)

tabela_features = pd.DataFrame(
    registros_features
)

np.save(
    CACHE / "features.npy",
    X,
)

tabela_features.to_csv(
    CACHE / "features.csv",
    index=False,
)

print(
    "Features:",
    X.shape,
)

display(
    tabela_features.groupby(
        ["classe", "split"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

## 8. Normalização e datasets balanceados

In [ ]:
indices_treino = np.where(
    tabela_features["split"].to_numpy()
    == "treino"
)[0]
indices_validacao = np.where(
    tabela_features["split"].to_numpy()
    == "validacao"
)[0]
indices_teste = np.where(
    tabela_features["split"].to_numpy()
    == "teste"
)[0]

scaler = StandardScaler()
X_treino = scaler.fit_transform(
    X[indices_treino]
).astype(np.float32)
X_validacao = scaler.transform(
    X[indices_validacao]
).astype(np.float32)
X_teste = scaler.transform(
    X[indices_teste]
).astype(np.float32)

np.savez_compressed(
    MODELOS / "normalizacao.npz",
    media=scaler.mean_.astype(np.float32),
    escala=scaler.scale_.astype(np.float32),
)

tabela_treino = (
    tabela_features.iloc[indices_treino]
    .copy()
    .reset_index(drop=True)
)
tabela_validacao = (
    tabela_features.iloc[indices_validacao]
    .copy()
    .reset_index(drop=True)
)
tabela_teste = (
    tabela_features.iloc[indices_teste]
    .copy()
    .reset_index(drop=True)
)


def rotulos(tabela):
    return {
        "evento": tabela["classe_id"]
        .astype(np.int32)
        .to_numpy(),
        "familia": tabela["familia"]
        .astype(np.int32)
        .to_numpy(),
        "perigo": tabela["perigo"]
        .astype(np.float32)
        .to_numpy()[:, None],
    }


y_treino = rotulos(tabela_treino)
y_validacao = rotulos(tabela_validacao)
y_teste = rotulos(tabela_teste)


def dataset_finito(X_dados, y_dados):
    return (
        tf.data.Dataset
        .from_tensor_slices((X_dados, y_dados))
        .batch(CFG.lote)
        .prefetch(1)
    )


def dataset_balanceado(
    X_dados,
    tabela,
    semente,
    normais_extras=None,
):
    datasets = []
    pesos = []

    for classe in CLASSES:
        indice = ID[classe]
        indices = np.where(
            tabela["classe_id"].to_numpy()
            == indice
        )[0]

        if (
            classe == "ambiente_normal"
            and normais_extras is not None
            and len(normais_extras) > 0
        ):
            indices = np.concatenate(
                [indices, normais_extras]
            )

        dados_rotulo = {
            "evento": tabela.iloc[indices][
                "classe_id"
            ].astype(np.int32).to_numpy(),
            "familia": tabela.iloc[indices][
                "familia"
            ].astype(np.int32).to_numpy(),
            "perigo": tabela.iloc[indices][
                "perigo"
            ].astype(np.float32).to_numpy()[:, None],
        }

        ds = (
            tf.data.Dataset
            .from_tensor_slices(
                (X_dados[indices], dados_rotulo)
            )
            .shuffle(
                min(len(indices), 5000),
                seed=semente + indice,
                reshuffle_each_iteration=True,
            )
            .repeat()
        )

        datasets.append(ds)
        pesos.append(
            PESOS_AMOSTRAGEM[classe]
        )

    return (
        tf.data.Dataset.sample_from_datasets(
            datasets,
            weights=pesos,
            seed=semente,
            stop_on_empty_dataset=False,
        )
        .batch(CFG.lote)
        .prefetch(1)
    )


ds_treino_1 = dataset_balanceado(
    X_treino,
    tabela_treino,
    CFG.seed,
)
ds_validacao = dataset_finito(
    X_validacao,
    y_validacao,
)
ds_teste = dataset_finito(
    X_teste,
    y_teste,
)

passos_epoca = int(
    np.clip(
        math.ceil(len(X_treino) / CFG.lote),
        450,
        1100,
    )
)

print("Treino:", X_treino.shape)
print("Validação:", X_validacao.shape)
print("Teste:", X_teste.shape)
print("Passos por época:", passos_epoca)

## 9. Modelo regularizado e macro-F1

In [ ]:
@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class SparseMacroF1(tf.keras.metrics.Metric):
    def __init__(
        self,
        num_classes,
        name="macro_f1",
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.num_classes = int(num_classes)
        self.matriz = self.add_weight(
            name="matriz_confusao",
            shape=(self.num_classes, self.num_classes),
            initializer="zeros",
            dtype=tf.float32,
        )

    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None,
    ):
        y_true = tf.cast(
            tf.reshape(y_true, [-1]),
            tf.int32,
        )
        y_pred = tf.argmax(
            y_pred,
            axis=-1,
            output_type=tf.int32,
        )
        matriz = tf.math.confusion_matrix(
            y_true,
            y_pred,
            num_classes=self.num_classes,
            dtype=tf.float32,
        )
        self.matriz.assign_add(matriz)

    def result(self):
        vp = tf.linalg.diag_part(self.matriz)
        fp = tf.reduce_sum(self.matriz, axis=0) - vp
        fn = tf.reduce_sum(self.matriz, axis=1) - vp
        precisao = tf.math.divide_no_nan(vp, vp + fp)
        recall = tf.math.divide_no_nan(vp, vp + fn)
        f1 = tf.math.divide_no_nan(
            2.0 * precisao * recall,
            precisao + recall,
        )
        presentes = tf.cast(
            tf.reduce_sum(self.matriz, axis=1) > 0,
            tf.float32,
        )
        return tf.math.divide_no_nan(
            tf.reduce_sum(f1 * presentes),
            tf.reduce_sum(presentes),
        )

    def reset_state(self):
        self.matriz.assign(
            tf.zeros_like(self.matriz)
        )

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_classes": self.num_classes,
        })
        return config


def criar_modelo():
    entrada = tf.keras.Input(
        shape=(X_treino.shape[1],),
        dtype=tf.float32,
        name="features",
    )

    x = tf.keras.layers.GaussianNoise(
        0.018
    )(entrada)
    x = tf.keras.layers.Dense(
        768,
        activation="swish",
        kernel_regularizer=(
            tf.keras.regularizers.l2(2e-5)
        ),
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.42)(x)
    x = tf.keras.layers.Dense(
        384,
        activation="swish",
        kernel_regularizer=(
            tf.keras.regularizers.l2(2e-5)
        ),
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(
        192,
        activation="swish",
        kernel_regularizer=(
            tf.keras.regularizers.l2(1e-5)
        ),
    )(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    evento = tf.keras.layers.Dense(
        len(CLASSES),
        activation="softmax",
        dtype=tf.float32,
        name="evento",
    )(x)
    familia = tf.keras.layers.Dense(
        3,
        activation="softmax",
        dtype=tf.float32,
        name="familia",
    )(x)
    perigo = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype=tf.float32,
        name="perigo",
    )(x)

    return tf.keras.Model(
        entrada,
        {
            "evento": evento,
            "familia": familia,
            "perigo": perigo,
        },
        name="aed_v7_yamnet_mfcc",
    )


def compilar(modelo, taxa):
    modelo.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=taxa,
            weight_decay=1e-5,
        ),
        loss={
            "evento": "sparse_categorical_crossentropy",
            "familia": "sparse_categorical_crossentropy",
            "perigo": "binary_crossentropy",
        },
        loss_weights={
            "evento": 1.0,
            "familia": 0.28,
            "perigo": 0.42,
        },
        metrics={
            "evento": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                ),
                SparseMacroF1(len(CLASSES)),
            ],
            "familia": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "perigo": [
                tf.keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.AUC(
                    name="auc"
                ),
            ],
        },
    )


modelo = criar_modelo()
compilar(modelo, 3e-4)
modelo.summary()

## 10. Treinamento em duas etapas

In [ ]:
MODELO_1 = MODELOS / "aed_v7_etapa_1.keras"
MODELO_2 = MODELOS / "aed_v7_etapa_2.keras"
MODELO_FINAL = MODELOS / "aed_v7_final.keras"


def callbacks(caminho, historico, paciencia):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            caminho,
            monitor="val_evento_macro_f1",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_evento_macro_f1",
            mode="max",
            patience=paciencia,
            restore_best_weights=False,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            RESULTADOS / historico
        ),
    ]


modelo.fit(
    ds_treino_1,
    validation_data=ds_validacao,
    epochs=CFG.epocas_etapa_1,
    steps_per_epoch=passos_epoca,
    callbacks=callbacks(
        MODELO_1,
        "historico_etapa_1.csv",
        CFG.paciencia,
    ),
    verbose=1,
)

modelo_1 = tf.keras.models.load_model(
    MODELO_1
)

# Mineração de ambientes normais confundidos com perigo.
saidas_treino = modelo_1.predict(
    X_treino,
    batch_size=256,
    verbose=1,
)

indices_normais = np.where(
    tabela_treino["classe_id"].to_numpy()
    == ID["ambiente_normal"]
)[0]

score_dificuldade = np.maximum(
    1.0 - saidas_treino["evento"][
        indices_normais,
        ID["ambiente_normal"],
    ],
    saidas_treino["perigo"]
    .reshape(-1)[indices_normais],
)

mascara = (
    score_dificuldade
    >= CFG.limiar_hard_negative
)
candidatos = indices_normais[mascara]
scores = score_dificuldade[mascara]
ordem = np.argsort(scores)[::-1]
hard_negatives = candidatos[
    ordem[:CFG.max_hard_negatives]
]
hard_extras = np.repeat(
    hard_negatives,
    CFG.repeticoes_hard_negative,
)

print(
    "Hard negatives selecionados:",
    len(hard_negatives),
)

hard_df = tabela_treino.iloc[
    hard_negatives
].copy()
hard_df["score_dificuldade"] = scores[
    ordem[:len(hard_negatives)]
]
hard_df.to_csv(
    RESULTADOS / "hard_negatives.csv",
    index=False,
)

ds_treino_2 = dataset_balanceado(
    X_treino,
    tabela_treino,
    CFG.seed + 100,
    normais_extras=hard_extras,
)

compilar(modelo_1, 6e-5)
modelo_1.fit(
    ds_treino_2,
    validation_data=ds_validacao,
    epochs=CFG.epocas_etapa_2,
    steps_per_epoch=passos_epoca,
    callbacks=callbacks(
        MODELO_2,
        "historico_etapa_2.csv",
        8,
    ),
    verbose=1,
)

modelo_2 = tf.keras.models.load_model(
    MODELO_2
)


def avaliar_validacao(modelo):
    saidas = modelo.predict(
        X_validacao,
        batch_size=CFG.lote,
        verbose=0,
    )
    previsto = np.argmax(
        saidas["evento"],
        axis=1,
    )
    real = y_validacao["evento"]
    return {
        "accuracy": float(
            accuracy_score(real, previsto)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                real,
                previsto,
            )
        ),
        "f1_macro": float(
            f1_score(
                real,
                previsto,
                average="macro",
            )
        ),
    }


validacao_1 = avaliar_validacao(modelo_1)
validacao_2 = avaliar_validacao(modelo_2)
comparacao = pd.DataFrame([
    {"etapa": "etapa_1", **validacao_1},
    {"etapa": "etapa_2", **validacao_2},
]).sort_values(
    ["f1_macro", "balanced_accuracy"],
    ascending=False,
).reset_index(drop=True)

display(comparacao)

etapa_selecionada = str(
    comparacao.loc[0, "etapa"]
)
modelo = (
    modelo_1
    if etapa_selecionada == "etapa_1"
    else modelo_2
)
modelo.save(MODELO_FINAL)
comparacao.to_csv(
    RESULTADOS / "comparacao_etapas.csv",
    index=False,
)

print(
    "Etapa selecionada:",
    etapa_selecionada,
)

## 11. Avaliação oficial no teste limpo

In [ ]:
saidas = modelo.predict(
    X_teste,
    batch_size=CFG.lote,
    verbose=1,
)

probs_evento = saidas["evento"]
pred_evento = np.argmax(
    probs_evento,
    axis=1,
)

real_evento = y_teste[
    "evento"
]

accuracy = accuracy_score(
    real_evento,
    pred_evento,
)

balanced = (
    balanced_accuracy_score(
        real_evento,
        pred_evento,
    )
)

f1_macro = f1_score(
    real_evento,
    pred_evento,
    average="macro",
)

relatorio = pd.DataFrame(
    classification_report(
        real_evento,
        pred_evento,
        target_names=CLASSES,
        output_dict=True,
        digits=4,
    )
).T

display(relatorio)

f1_minimo = float(
    relatorio.loc[
        CLASSES,
        "f1-score",
    ].min()
)

atingiu_meta = bool(
    accuracy >= CFG.meta_accuracy
    and f1_macro
    >= CFG.meta_f1_macro
    and f1_minimo
    >= CFG.meta_f1_classe
)

resumo = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(
        balanced
    ),
    "f1_macro": float(
        f1_macro
    ),
    "menor_f1_classe": float(
        f1_minimo
    ),
    "atingiu_meta": (
        atingiu_meta
    ),
    "etapa_selecionada": (
        etapa_selecionada
    ),
    "auditoria_split": (
        auditoria_split
    ),
    "metas": {
        "accuracy": (
            CFG.meta_accuracy
        ),
        "f1_macro": (
            CFG.meta_f1_macro
        ),
        "f1_classe": (
            CFG.meta_f1_classe
        ),
    },
}

print(
    json.dumps(
        resumo,
        indent=2,
    )
)

relatorio.to_csv(
    RESULTADOS
    / "metricas_por_classe.csv"
)

(
    RESULTADOS
    / "resumo_metricas.json"
).write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

matriz = confusion_matrix(
    real_evento,
    pred_evento,
)

plt.figure(figsize=(10, 8))
plt.imshow(
    matriz,
    cmap="Blues",
)
plt.colorbar()
plt.title(
    "Matriz de confusão — AED v7"
)
plt.xticks(
    range(len(CLASSES)),
    CLASSES,
    rotation=45,
    ha="right",
)
plt.yticks(
    range(len(CLASSES)),
    CLASSES,
)

for linha in range(len(CLASSES)):
    for coluna in range(len(CLASSES)):
        plt.text(
            coluna,
            linha,
            str(matriz[linha, coluna]),
            ha="center",
            va="center",
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()

plt.savefig(
    RESULTADOS
    / "matriz_confusao.png",
    dpi=170,
)

plt.show()

## 12. Avaliação por gravação

In [ ]:
tabela_teste = (
    tabela_features.iloc[
        indices_teste
    ]
    .copy()
    .reset_index(drop=True)
)

for indice, classe in enumerate(
    CLASSES
):
    tabela_teste[
        f"prob_{classe}"
    ] = probs_evento[:, indice]

linhas = []

for grupo, parte in tabela_teste.groupby(
    "grupo"
):
    vetor = np.asarray(
        [
            parte[
                f"prob_{classe}"
            ].mean()
            for classe in CLASSES
        ]
    )

    real = int(
        parte["classe_id"]
        .mode()
        .iloc[0]
    )

    previsto = int(
        np.argmax(vetor)
    )

    linhas.append(
        {
            "grupo": grupo,
            "real": real,
            "previsto": previsto,
            "real_nome": CLASSES[real],
            "previsto_nome": (
                CLASSES[previsto]
            ),
            "confianca": float(
                vetor[previsto]
            ),
        }
    )

gravacoes = pd.DataFrame(
    linhas
)

metricas_gravacao = {
    "accuracy": float(
        accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "f1_macro": float(
        f1_score(
            gravacoes["real"],
            gravacoes["previsto"],
            average="macro",
        )
    ),
}

print(
    json.dumps(
        metricas_gravacao,
        indent=2,
    )
)

gravacoes.to_csv(
    RESULTADOS
    / "predicoes_por_gravacao.csv",
    index=False,
)

## 13. Avaliação externa AOB corrigida

In [ ]:
if externo.empty:
    print("Dataset-AOB externo não disponível.")
else:
    vetores_externos = []
    linhas_externas = []

    for linha in tqdm(
        externo.itertuples(),
        total=len(externo),
        desc="AOB externo",
    ):
        try:
            audio = carregar_audio(
                linha.arquivo
            )
            janelas = (
                janelas_onset(audio)
                if linha.classe in RARAS
                else janelas_uniformes(audio)
            )

            for janela in janelas:
                vetores_externos.append(
                    extrair_vetor(janela)
                )
                linhas_externas.append({
                    "classe": linha.classe,
                    "classe_id": linha.classe_id,
                    "grupo": linha.grupo,
                })
        except Exception as erro:
            print(
                "Falha externa:",
                linha.arquivo,
                erro,
            )

    if vetores_externos:
        X_externo = scaler.transform(
            np.asarray(
                vetores_externos,
                dtype=np.float32,
            )
        ).astype(np.float32)

        tabela_externa = pd.DataFrame(
            linhas_externas
        )
        saidas_externas = modelo.predict(
            X_externo,
            batch_size=CFG.lote,
            verbose=1,
        )
        pred_externo = np.argmax(
            saidas_externas["evento"],
            axis=1,
        )
        real_externo = tabela_externa[
            "classe_id"
        ].to_numpy()
        presentes = sorted(
            np.unique(real_externo).tolist()
        )

        metricas_externas = {
            "accuracy": float(
                accuracy_score(
                    real_externo,
                    pred_externo,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    real_externo,
                    pred_externo,
                )
            ),
            "f1_macro_classes_presentes": float(
                f1_score(
                    real_externo,
                    pred_externo,
                    labels=presentes,
                    average="macro",
                    zero_division=0,
                )
            ),
            "classes_presentes": [
                CLASSES[indice]
                for indice in presentes
            ],
            "predicoes_fora_das_classes_presentes": int(
                np.sum(
                    ~np.isin(
                        pred_externo,
                        presentes,
                    )
                )
            ),
        }

        print(
            json.dumps(
                metricas_externas,
                ensure_ascii=False,
                indent=2,
            )
        )

        relatorio_externo = pd.DataFrame(
            classification_report(
                real_externo,
                pred_externo,
                labels=presentes,
                target_names=[
                    CLASSES[indice]
                    for indice in presentes
                ],
                output_dict=True,
                zero_division=0,
            )
        ).T
        display(relatorio_externo)
        relatorio_externo.to_csv(
            RESULTADOS
            / "metricas_externas_aob.csv"
        )
        (
            RESULTADOS
            / "metricas_externas_aob.json"
        ).write_text(
            json.dumps(
                metricas_externas,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

## 14. Exportar classificador e verificar paridade

In [ ]:
TFLITE_FLOAT32 = (
    MODELOS
    / "aed_v7_classificador_float32.tflite"
)
TFLITE_FLOAT16 = (
    MODELOS
    / "aed_v7_classificador_float16.tflite"
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(modelo)
)
TFLITE_FLOAT32.write_bytes(
    conversor.convert()
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(modelo)
)
conversor.optimizations = [
    tf.lite.Optimize.DEFAULT
]
conversor.target_spec.supported_types = [
    tf.float16
]
# Explicitly allow TensorFlow operations (flex ops) for float16 conversion
conversor.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
TFLITE_FLOAT16.write_bytes(
    conversor.convert()
)


def prever_tflite(caminho, dados):
    interpretador = tf.lite.Interpreter(
        model_path=str(caminho)
    )
    interpretador.allocate_tensors()
    entrada = interpretador.get_input_details()[0]
    saidas_info = interpretador.get_output_details()
    eventos = []
    perigos = []

    for vetor in dados:
        interpretador.set_tensor(
            entrada["index"],
            vetor[None, ...].astype(
                entrada["dtype"]
            ),
        )
        interpretador.invoke()
        valores = [
            interpretador.get_tensor(
                item["index"]
            )
            for item in saidas_info
        ]

        evento = next(
            valor
            for valor in valores
            if valor.shape[-1]
            == len(CLASSES)
        )
        perigo = next(
            valor
            for valor in valores
            if valor.shape[-1] == 1
        )
        eventos.append(evento[0])
        perigos.append(float(perigo[0][0]))

    return (
        np.asarray(eventos),
        np.asarray(perigos),
    )


rng = np.random.default_rng(CFG.seed)
quantidade = min(250, len(X_teste))
indices = rng.choice(
    len(X_teste),
    size=quantidade,
    replace=False,
)
amostra = X_teste[indices]
keras_saida = modelo.predict(
    amostra,
    batch_size=64,
    verbose=0,
)

linhas = []
for caminho in (
    TFLITE_FLOAT32,
    TFLITE_FLOAT16,
):
    evento_tflite, perigo_tflite = (
        prever_tflite(
            caminho,
            amostra,
        )
    )
    linhas.append({
        "modelo": caminho.name,
        "concordancia_evento": float(
            np.mean(
                np.argmax(
                    evento_tflite,
                    axis=1,
                )
                == np.argmax(
                    keras_saida["evento"],
                    axis=1,
                )
            )
        ),
        "concordancia_perigo_050": float(
            np.mean(
                (perigo_tflite >= 0.50)
                == (
                    keras_saida["perigo"]
                    .reshape(-1)
                    >= 0.50
                )
            )
        ),
        "diferenca_max_evento": float(
            np.max(
                np.abs(
                    evento_tflite
                    - keras_saida["evento"]
                )
            )
        ),
    })

paridade = pd.DataFrame(linhas)
display(paridade)
paridade.to_csv(
    RESULTADOS / "paridade_tflite.csv",
    index=False,
)

if (
    paridade["concordancia_evento"].min()
    < CFG.meta_paridade
):
    raise RuntimeError(
        "Paridade TFLite abaixo da meta."
    )

(
    MODELOS / "configuracao_aed_v7.json"
).write_text(
    json.dumps(
        {
            "classes": CLASSES,
            "configuracao": asdict(CFG),
            "yamnet_handle": YAMNET_HANDLE,
            "dimensao_features": int(
                X_treino.shape[1]
            ),
            "resultado": resumo,
            "resultado_gravacao": (
                metricas_gravacao
            ),
            "etapa_selecionada": (
                etapa_selecionada
            ),
            "auditoria_split": (
                auditoria_split
            ),
            "atingiu_meta": atingiu_meta,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# Baixa o extrator YAMNet em TFLite para completar o pacote embarcado.
YAMNET_TFLITE = MODELOS / "yamnet.tflite"
YAMNET_TFLITE_URL = (
    "https://tfhub.dev/google/lite-model/"
    "yamnet/tflite/1?lite-format=tflite"
)

try:
    resposta_yamnet = requests.get(
        YAMNET_TFLITE_URL,
        timeout=180,
        allow_redirects=True,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
    )
    resposta_yamnet.raise_for_status()
    YAMNET_TFLITE.write_bytes(
        resposta_yamnet.content
    )

    if YAMNET_TFLITE.stat().st_size < 1_000_000:
        raise RuntimeError(
            "O arquivo YAMNet baixado é inválido."
        )

    teste_yamnet = tf.lite.Interpreter(
        model_path=str(YAMNET_TFLITE)
    )
    teste_yamnet.allocate_tensors()

    print(
        "YAMNet TFLite incluído:",
        YAMNET_TFLITE,
    )
except Exception as erro:
    YAMNET_TFLITE.unlink(
        missing_ok=True
    )
    print(
        "Não foi possível incluir o YAMNet TFLite:",
        erro,
    )

README_IMPLANTACAO = MODELOS / "README_IMPLANTACAO.txt"
README_IMPLANTACAO.write_text(
    """
AED v7 — ordem de inferência na Raspberry Pi

1. Capturar áudio mono em 16 kHz.
2. Trabalhar com janelas de 2 segundos.
3. Executar yamnet.tflite para obter embeddings.
4. Calcular as mesmas estatísticas YAMNet e MFCC do notebook.
5. Normalizar com normalizacao.npz.
6. Executar aed_v7_classificador_float16.tflite.
7. Aplicar a regra temporal de duas janelas consecutivas.

O VAD não bloqueia o AED. Ele somente aciona Whisper e NLP.
""".strip(),
    encoding="utf-8",
)


## 15. Pacote final

In [ ]:
if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in MODELOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

for arquivo in RESULTADOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

ZIP_FINAL = shutil.make_archive(
    "/content/aed_v7_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print(
    "Meta atingida:",
    atingiu_meta,
)
print(
    "ZIP:",
    ZIP_FINAL,
)

files.download(
    ZIP_FINAL
)

## Observação de implantação

O classificador v7 ainda usa duas partes:

1. YAMNet para gerar embeddings da janela de áudio;
2. `aed_v7_classificador_float16.tflite` para classificar as features.

O pacote inclui `normalizacao.npz`, a configuração e as métricas.
O VAD continua separado e somente aciona Whisper/NLP; ele não bloqueia o AED.